In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import time
from datetime import datetime

import networkx as nx
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader


from tensorboardX import SummaryWriter
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

import matplotlib.font_manager as fm
from matplotlib import font_manager
import matplotlib.ticker as ticker

batch_size = 128

In [ ]:
# Use LaTeX-style fonts for professional look
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 24,  # Adjust based on target journal/conference
    "axes.labelsize": 26,
    "axes.titlesize": 24,
    "legend.fontsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "lines.linewidth": 1.5,  # Thicker lines for visibility
    "lines.markersize": 6,
    "grid.linestyle": "--",  # Dashed grid for subtlety
    "grid.alpha": 0.5,  # Slight transparency for readability
    "legend.frameon": False,  # No box around legends
    "figure.dpi": 300,  # High resolution
    "savefig.dpi": 300,  # High-resolution output
    "text.usetex": False,  # Use LaTeX for better typography (if available)
    "axes.grid": True,  # Enable grid
    "axes.spines.top": True,  # Hide top spine
    "axes.spines.right": True,  # Hide right spine
})

### Loading strain and stiffness reduction data

In [ ]:
stiffness_data_path = 'Data/Stiffness_Reduction'
strain_data_path = 'Data/Strain'

# Stiffness Data
stiff_file_paths = [f.path for f in os.scandir(stiffness_data_path) if f.path.endswith('.h5')]
stiff_file_paths.sort()
stiffness_dfs = {}
for i, file_path in enumerate(stiff_file_paths):
    stiffness_dfs[f'df{i}'] = pd.read_hdf(file_path)['Stiffness']

# Strain Data
strain_file_paths = [f.path for f in os.scandir(strain_data_path) if f.path.endswith('.h5')]
strain_file_paths.sort()
strain_dfs = {}
for i, file_path in enumerate(strain_file_paths):
    strain_dfs[f'df{i}'] = pd.read_hdf(file_path)



In [ ]:
def resample_stiffness_to_match_strain(strain_df, stiffness_df):
    strain_length = len(strain_df)
    stiffness_length = len(stiffness_df)
    
    # Assuming strain_df has a column with the strain values (e.g., 'strain')
    # and stiffness_df has the corresponding stiffness values in one or more columns.

    if strain_length > stiffness_length:
        # Interpolation: Upsample stiffness_df to match strain_df length
        # Assuming both dataframes are 1D for now, you can extend this to multiple columns later.
        x_old = np.linspace(0, 1, stiffness_length)  # Normalized index for stiffness
        x_new = np.linspace(0, 1, strain_length)  # Normalized index for strain

        # Interpolating f stiffness_df
        stiffness_df_resampled = pd.DataFrame(np.interp(x_new, x_old, stiffness_df))
    
    elif strain_length < stiffness_length:
        # Downsampling: Downsample stiffness_df to match strain_df length
        x_old = np.linspace(0, 1, stiffness_length)  # Normalized index for stiffness
        x_new = np.linspace(0, 1, strain_length)  # Normalized index for strain

        # Find the closest indices in stiffness_df to the new sample points
        idx_new = np.searchsorted(x_old, x_new)
        idx_new = np.clip(idx_new, 0, stiffness_length - 1)  # Ensure indices are valid

        stiffness_df_resampled = stiffness_df.iloc[idx_new].reset_index(drop=True)
    
    else:
        # If already the same length, no action required
        stiffness_df_resampled = stiffness_df.reset_index(drop=True)
    
    return stiffness_df_resampled


def percentage_change_from_max_normal(stiffness_df):
    if isinstance(stiffness_df, pd.Series):
        max_value = stiffness_df.max()
        percentage_change_df = (stiffness_df / max_value) * 100
        return percentage_change_df
    elif isinstance(stiffness_df, pd.DataFrame):
        max_value = stiffness_df.max().max()
        percentage_change_df = (stiffness_df / max_value) * 100
        return percentage_change_df
    else:
        raise ValueError("Input must be a pandas DataFrame or Series")
    

########## Correcting starting stiffness values ##########
def percentage_change_from_max(stiffness_df):
    if isinstance(stiffness_df, pd.Series):
        max_index = stiffness_df.idxmax()
        max_value = stiffness_df[max_index]
        percentage_change_df = (stiffness_df / max_value) * 100
        percentage_change_df.loc[:max_index] = 100  # Ensure correct assignment
        return percentage_change_df

    elif isinstance(stiffness_df, pd.DataFrame):
        percentage_change_df = stiffness_df.copy()
        max_values = stiffness_df.max()  # Get max for each column
        
        for col in stiffness_df.columns:
            max_idx_col = stiffness_df[col].idxmax()
            percentage_change_df[col] = (stiffness_df[col] / max_values[col]) * 100  # Normalize per column
            percentage_change_df.loc[:max_idx_col, col] = 100  # Set values before max to 100
        
        return percentage_change_df
    else:
        raise ValueError("Input must be a pandas DataFrame or Series")

In [ ]:
last_cycle = {}

for key in stiffness_dfs.keys():
    last_cycle[key] = len(stiffness_dfs[key])

last_cycle

In [ ]:
#### Resample Strain, Smooth Strain and Stiffness Daata, and Match the Time Stamps ####

stiffness_post = {}
strain_post = {}
target_indexes = {}
stiffness_to_find_index = {}
# Use the key from strain_dfs
for key, strain_df in strain_dfs.items():
    
    if key == 'df2':
        strain_df = strain_df.iloc[:,:-8]
        
    # Resample the strain data and smooth it with rolling mean
    strain_resampled = strain_df.resample("200s").mean().rolling(10).mean()

    # Drop the NaN values
    strain_resampled = strain_resampled.dropna()


    ##### Custom Feature Engineering #####
    ################################################
    # strain_temp= np.cumsum(abs(np.diff(strain_resampled, axis=0)), axis=0)
    # strain_temp= pd.DataFrame(strain_temp)
    # # copy index from strain_resampled
    # strain_temp.index = strain_resampled.iloc[1:,:].index
    # strain_resampled = strain_temp
    # #strain_resampled = strain_resampled.dropna()
    #####################################################
    
    # Store the resampled strain in strain_post
    strain_post[key] = strain_resampled
    
    # Get the corresponding stiffness_df using the same key from stiffness_dfs
    stiffness_df = stiffness_dfs[key].rolling(50).mean()

    # Drop the NaN values
    stiffness_df = stiffness_df.dropna()

    # Calculate the percentage change from the maximum value
    stiffness_df = percentage_change_from_max(stiffness_df)
    
    # Resample the stiffness data to match the strain
    stiffness_resampled = resample_stiffness_to_match_strain(strain_resampled, stiffness_df)

    stiffness_to_find_index[key] = stiffness_resampled.copy()

    
    # Store the resampled stiffness in stiffness_post
    stiffness_post[key] = pd.DataFrame(stiffness_resampled)

    stiffness_post[key].index = strain_post[key].index

    # #### RUL ESTIMATION ###########
    #stiffness_post[key] = pd.DataFrame(np.linspace(stiffness_post[key].index.total_seconds()[-1], 0, len(stiffness_post[key])))

    
    # # #########################################
    
    stiffness_post[key].index = strain_post[key].index

    # # print the NaN values in the stiffness and strain data
    # print(f"NaN values in {key}:")
    # print(f"Stiffness: {stiffness_post[key].isna().sum().sum()}")
    # print(f"Strain: {strain_post[key].isna().sum().sum()}")

In [ ]:
# plot the strain
plt.figure(figsize=(10, 6))
plt.plot(strain_post['df3'], label='Strain')
plt.xlabel('Time')
plt.ylabel('Strain')
plt.title('Strain Data')
plt.legend()
plt.show()

In [ ]:
target_indexes = {}

def find_closest_index(array, target):
    # Find index of the closest value to the target in the array
    return np.abs(array - target).argmin()

for key, values in stiffness_post.items():
    # Convert the list of stiffness values to a NumPy array for efficient operations
    stiffness_values = np.array(values)
    
    # Find the closest index of the value 100
    closest_index_99 = find_closest_index(stiffness_values, 99)
    value_99 = stiffness_values[closest_index_99]
    

    # Initialize variables
    index_99 = None
    valid_95 = []
    valid_90 = []
    valid_85 = []

  
    index_99 = closest_index_99
    # Filter out values before the index_99
    filtered_values = stiffness_values[index_99 + 1:]
   
    
# Find the closest index in the filtered array
    target_indexes[key] = {
        99: index_99,
        95: find_closest_index(filtered_values, 95) + index_99 + 1,
        90: find_closest_index(filtered_values, 90) + index_99 + 1,
        85: find_closest_index(filtered_values, 70) + index_99 + 1
    }

In [ ]:
# Plot stiffness for all FODs
strain_x_rescaled = {}
stiffness_x_rescaled = {}

plt.figure(figsize=(12, 6))
for key, _ in strain_post.items():
    # Retrieve original x values
    strain_x = strain_post[key].index.total_seconds()
    stiffness_x = stiffness_post[key].index.total_seconds()
    
    # Scale the x-axis so the last point corresponds to last_cycle[key]
    max_time = max(strain_x.max(), stiffness_x.max())
    strain_x_rescaled[key] = strain_x * (last_cycle[key] / max_time)
    stiffness_x_rescaled[key] = stiffness_x * (last_cycle[key] / max_time)
    
    # Plot stiffness data
    plt.scatter(stiffness_x_rescaled[key], stiffness_post[key], label=f"FOD{int(key.split('f')[-1])+3}", s=1)
    
    # Customize legend and axes
    plt.legend(loc='best', fontsize='x-small', ncol=2)
    plt.title(f"Stiffness vs Cycles")
    plt.xlabel(f"Cycles")
    plt.ylabel("Normalized stiffness")
    
plt.show()

In [ ]:
# Drop

drop = 85

for key, values in stiffness_post.items():
    # Drop values before the target index
    stiffness_post[key] = values[:target_indexes[key][drop]]
    strain_post[key] = strain_post[key].iloc[:target_indexes[key][drop]]
    stiffness_x_rescaled[key] = stiffness_x_rescaled[key][:target_indexes[key][drop]]
    strain_x_rescaled[key] = strain_x_rescaled[key][:target_indexes[key][drop]]

    plt.figure(figsize=(12, 6))

    # Plot stiffness data
    plt.scatter(stiffness_x_rescaled[key], stiffness_post[key], label=f"FOD{int(key.split('f')[-1])+3}", s=1)
    
    # Customize legend and axes
    plt.legend(loc='best', fontsize='x-small', ncol=2)
    plt.title(f"Stiffness vs Strain")
    plt.xlabel(f"Time (s)")
    plt.ylabel("Normalized stiffness")
        
    plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
for key, _ in strain_post.items():
    # Retrieve original x values
    cut_index = target_indexes[key][drop]
    strain_x_rescaled[key] = strain_x_rescaled[key][:cut_index+1] 
    stiffness_x_rescaled[key] = stiffness_x_rescaled[key][:cut_index+1]

    # Plot stiffness data
    plt.scatter(stiffness_x_rescaled[key], stiffness_post[key], label=f"FOD{int(key.split('f')[-1])+3}", s=1)
    
    # Customize legend and axes
    plt.legend(loc='best', fontsize='x-small', ncol=2, markerscale=3)
    plt.title(f"Stiffness reduction per cycles")
    plt.xlabel(f"Cycles")
    plt.ylabel("Stiffness (%)")
    
plt.show()

In [ ]:
# Leave out FOD3 sinnce it has only 6 sensors

strain_data = []
stiffness_data = []

for key, _ in strain_post.items():
    if key == 'df0':
        continue
    strain_data.append(strain_post[key].values)
    
    # Reshape stiffness data to ensure it is (N, 1)
    stiffness_values = stiffness_post[key].values
    if stiffness_values.ndim == 1:
        stiffness_values = stiffness_values.reshape(-1, 1)
    
    stiffness_data.append(stiffness_values)


In [ ]:
for strain , stiffness in zip(strain_data, stiffness_data):
    print(strain.shape, stiffness.shape)
    

## Model

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, activation, seed=None, initialize_weights=None):
        """
        Arguments
        ---------
        input_dim : int
            Dimension of the input vector
        output_dim : int
            Dimension of the output vector/prediction
        width : int
            Width of each hidden layer (number of neurons per layer)
        depth : int
            Depth of the neural network (number of hidden layers + output layer)
        activation : torch.nn.Module
            Type of activation function used in each neuron
        seed : int
            Random seed for keep weights and biases initialization constant to detect the effect of hyperparameters
        """
        super(MLP, self).__init__()
        
        if seed is not None:       # Set the seed if provided for reproducibility in weights and biases initialization
            self.set_seed(seed)
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.width = width
        self.depth = depth
        self.activation = activation
        self.loss= nn.MSELoss()
        
        # Define the input layer
        self.input_layer = nn.Linear(input_dim, width, bias=True)
        self.layer_norm = nn.LayerNorm(width)
        self.dropout = nn.Dropout(0.2)

        # Define the hidden layers
        self.hidden_layers = nn.ModuleList()
        for i in range(depth-1):
            self.hidden_layers.append(nn.Linear(width, width, bias=True))
            self.hidden_layers.append(self.dropout)
            #self.hidden_layers.append(nn.LayerNorm(width))

        # Define the output layer
        self.output_layer = nn.Linear(width, output_dim, bias=True)

        # Initialize the weights and biases of the network
        if initialize_weights is not None:
            self._initialize_weights()
        

    # Function to set the seed for CPU and GPU
    def set_seed(self, seed):
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    

    # Function to initialize the weights and biases of the network
    def _initialize_weights(self):
        # Apply uniform initialization to all weights and biases of each layer from (-1, 1)
        for layer in self.children():
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, mean=0, std=1)
                nn.init.normal_(layer.bias, mean=0, std=1)

    # Function of forward pass of the neural network
    def forward(self, x):
        x = self.input_layer(x)
        #x = self.layer_norm(x)
        x = self.activation(x)
        for layer in self.hidden_layers:
            x = layer(x)
            x = self.activation(x)
        x = self.output_layer(x)
        return x

# print lernable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# build model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(input_dim=16, output_dim=1, width=16, depth=3, activation=nn.ReLU(), seed=None, initialize_weights=False).to(device)
opt = optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
# Learning Rate Scheduler based on training loss
scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.8, patience=10)
print(model)
print(f"Model has {count_parameters(model):,} trainable parameters.")
# print the available data points
temp_len = 0
for data in strain_data:
    temp_len += len(data)
print(f'Total data points: {temp_len}')


## Training Loop

In [ ]:
def weighted_mse_loss(pred, target, weight_range=(0.2, 0.5), weight_value=2.0):
    """
    Weighted MSE loss giving higher importance to target values within a specific range.
    
    Parameters:
        pred (Tensor): Predicted values (batch_size, num_outputs).
        target (Tensor): Target values (batch_size, num_outputs).
        weight_range (tuple): Range of target values to apply the weight.
        weight_value (float): Weight factor to apply to the loss for targets within the range.
    
    Returns:
        Tensor: Weighted MSE loss.
    """
    # Calculate the standard MSE loss
    mse_loss = F.mse_loss(pred, target, reduction='none')  # Keep per-element loss
    
    # Create a weight mask
    weight_mask = (target >= weight_range[0]) & (target <= weight_range[1])
    weights = torch.ones_like(target)
    weights[weight_mask] = weight_value  # Apply higher weight to values in the range
    
    # Apply weights to the loss
    weighted_loss = mse_loss * weights
    return weighted_loss.mean()  # Reduce to a single loss value

In [ ]:
def train(train_loader, val_loader, writer, epochs, patience=20):
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        for data in train_loader:
            x , y = data
            x = x.to(device)
            y = y.to(device)
            opt.zero_grad()
            pred = model(x)
            # Use the weighted loss function
            loss = weighted_mse_loss(pred, y, weight_range=(0.0, 0.80), weight_value=6.0)
            #loss = model.loss(pred, data.y)
            loss.backward()
            opt.step()

            total_loss += loss.item()

        total_loss /= len(train_loader.dataset)
        train_losses.append(total_loss)
        writer.add_scalar("loss", total_loss, epoch)

        val_mse = validation(val_loader, model)
        val_losses.append(val_mse)
        print(f"Epoch {epoch}. Loss: {total_loss:.4f}. Val MSE: {val_mse:.4f}")
        writer.add_scalar("val_mse", val_mse, epoch)

        scheduler.step(total_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch}. Current learning rate: {current_lr:.6f}")
        writer.add_scalar("learning_rate", current_lr, epoch)

        if epoch < 3:
            continue
        else:
            if val_mse < best_val_loss:
                best_val_loss = val_mse
                best_model_state = model.state_dict()
                torch.save(best_model_state, "best_model/best_model_state.pth")
                print(f"New best model found at epoch {epoch} with Val MSE: {val_mse:.4f}")
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                print(f"No improvement at epoch {epoch}. Patience: {epochs_without_improvement}/{patience}")

        if epochs_without_improvement >= patience:
            print(f"Early stopping after {epoch} epochs. Best Val MSE: {best_val_loss:.4f}")
            break

    if best_model_state is not None:
        model.load_state_dict(torch.load("best_model/best_model_state.pth", weights_only=True))
        print("Loaded the best model state.")
    else:
        print("No best model state was saved.")

    return model, train_losses, val_losses

def validation(val_loader, model):
    model.eval()
    total_loss = 0

    for data in val_loader:
        with torch.no_grad():
            x , y = data
            x = x.to(device)
            y = y.to(device)

            pred = model(x)
            loss = model.loss(pred, y)
            total_loss += loss.item() 

    return total_loss / len(val_loader.dataset)

In [ ]:
# FOD 4 is df1 and specimen_data[0]
# FOD 5 is df2 and specimen_data[1]
# FOD 6 is df3 and specimen_data[2]
# FOD 7 is df4 and specimen_data[3]

# -------------------------------------------------------------

test_key = 'df1' # Corresponds to index 3 if using 0-based indexing

# 1. Concatenate the training data (using numpy initially)
train_data_np = np.concatenate((strain_data[1], strain_data[2], strain_data[3]), axis=0)
train_target_np = np.concatenate((stiffness_data[1], stiffness_data[2], stiffness_data[3]), axis=0)

# 2. Prepare validation data (using numpy initially)
val_data_np = strain_data[0] # Assuming index 3 corresponds to 'df4'
val_target_np = stiffness_data[0]

# --- Normalization Steps START ---

# 3. Calculate mean and std deviation *only* from the training input data (features)
input_mean = np.mean(train_data_np, axis=0, keepdims=True) # Keepdims for broadcasting
input_std = np.std(train_data_np, axis=0, keepdims=True)   # Keepdims for broadcasting

# Add a small epsilon to std deviation to prevent division by zero in case of constant features
epsilon = 1e-8
input_std_eps = input_std + epsilon

# 4. Apply normalization (standardization) to training and validation input data
# Formula: Z = (X - mean) / std
train_data_normalized_np = (train_data_np - input_mean) / input_std_eps
val_data_normalized_np = (val_data_np - input_mean) / input_std_eps # Use TRAINING mean/std

# 5. Calculate min and max for the training target data (as indicated by norm_params)
#    Note: This calculates parameters but doesn't apply target normalization here.
#          If you want Min-Max scaling for targets, apply it similarly:

target_min = np.min(train_target_np, axis=0)
target_max = np.max(train_target_np, axis=0)

target_range = target_max - target_min + epsilon
train_target_normalized_np = (train_target_np - target_min) / target_range
val_target_normalized_np = (val_target_np - target_min) / target_range



# 6. Save normalization parameters
norm_params = {
    'input_mean': input_mean.flatten(), # Flatten since keepdims=True made them 2D
    'input_std': input_std.flatten(),   # Flatten
    'target_min': target_min,           # Min/Max are often used as is
    'target_max': target_max
}

print("Normalization Parameters calculated from Training Set:")
print(f"Input Mean: {norm_params['input_mean']}")
print(f"Input Std Dev: {norm_params['input_std']}")
print(f"Target Min: {norm_params['target_min']}")
print(f"Target Max: {norm_params['target_max']}")
print("-" * 30)

# --- Normalization Steps END ---


# 7. Create PyTorch tensors from the *normalized* input data and original target data
train_data = torch.tensor(train_data_normalized_np, dtype=torch.float32)
train_target = torch.tensor(train_target_normalized_np, dtype=torch.float32) # Using original targets
train_dataset = torch.utils.data.TensorDataset(train_data, train_target)

val_data = torch.tensor(val_data_normalized_np, dtype=torch.float32)
val_target = torch.tensor(val_target_normalized_np, dtype=torch.float32) # Using original targets
val_dataset = torch.utils.data.TensorDataset(val_data, val_target)

print("Data shapes after processing:")
print(f"Train data tensor shape: {train_data.shape}")
print(f"Train target tensor shape: {train_target.shape}")
print(f"Validation data tensor shape: {val_data.shape}")
print(f"Validation target tensor shape: {val_target.shape}")
print("-" * 30)

# --- Optional: Create DataLoaders ---
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Created Train DataLoader with {len(train_loader)} batches.")
print(f"Created Validation DataLoader with {len(val_loader)} batches.")

writer = SummaryWriter("./log/" + datetime.now().strftime("%Y%m%d-%H%M%S"))

model, train_losses, val_losses = train(train_loader, val_loader, writer,epochs=1000,  patience=80)

fig_width = 16
fig_height = 9
plt.figure(figsize=(fig_width, fig_height))
plt.semilogy(train_losses, label='Training Loss')  # Use semilogy for logarithmic y-axis
plt.semilogy(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, which="both", ls="--")  # Add grid lines for both major and minor ticks
plt.show()


## Inference

In [ ]:
import torch
import numpy as np

def inference(model, dataset, device, key, norm_params):
    model.eval()
    true_values_unnormalized = []
    predicted_values_unnormalized = []

    all_true = []
    all_pred = []
    
    with torch.no_grad():
        for data in dataset:
            x, y = data
            x = x.to(device)
            y = y.to(device)
            pred = model(x)
            
            # Normalized true and predicted values
            true_val_norm = y.cpu().numpy().flatten()
            pred_val_norm = pred.cpu().numpy().flatten()

            # Unnormalize the data
            true_val_unnorm = true_val_norm * (norm_params['target_max'] - norm_params['target_min']) + norm_params['target_min']
            pred_val_unnorm = pred_val_norm * (norm_params['target_max'] - norm_params['target_min']) + norm_params['target_min']

            # Store unnormalized values
            true_values_unnormalized.append(true_val_unnorm)
            predicted_values_unnormalized.append(pred_val_unnorm)

            # Collect all values for final metric calculation
            all_true.extend(true_val_unnorm)
            all_pred.extend(pred_val_unnorm)

    # Convert lists to numpy arrays
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    # Compute final metrics
    mse = np.mean((all_true - all_pred) ** 2)
    rmse = np.sqrt(mse)
    
    # Avoid division by zero in MAPE
    nonzero_mask = all_true != 0
    if np.any(nonzero_mask):
        avg_mape = np.mean(np.abs((all_true[nonzero_mask] - all_pred[nonzero_mask]) / all_true[nonzero_mask]) * 100)
    else:
        avg_mape = float('inf')  # Undefined MAPE when all true values are zero

    return true_values_unnormalized, predicted_values_unnormalized, mse, rmse, avg_mape



def plot_predictions(true_values, predicted_values, mse, rmse, mape, title, key='df3'):
    # Define figure size for 16:9 aspect ratio
    fig_width = 16
    fig_height = 9
    plt.figure(figsize=(fig_width, fig_height))
    
    plt.plot(strain_x_rescaled[key], true_values, label="True Values", color="b")
    plt.plot(strain_x_rescaled[key], predicted_values, label="Predicted Values", color="r", linestyle='--')
    
    # Create metrics text with proper alignment using monospace font
    metrics_text = (
        f"MSE:  {mse:.2f} \n"
        f"RMSE: {rmse:.2f} \n"
        f"MAPE: {mape:.2f}%"
    )
    
    # Position the text box in the upper right corner with better styling
    plt.annotate(metrics_text, xy=(0.95, 0.95), xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.6", facecolor='white', alpha=0.8, 
                           edgecolor='gray', linewidth=1),
                 ha='right', va='top', fontsize=14, family='monospace')
    
    plt.xlabel("Cycles")
    plt.ylabel("Stiffness (%)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

def enable_dropout(model):
    """ Function to enable the dropout layers during test-time """
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.train()

def mc_dropout_inference(model, dataset, device, norm_params, num_samples=100, key='df3'):
    model.eval()
    enable_dropout(model)  # Enable dropout during inference
    
    true_values_unnormalized = []
    mc_predicted_values_unnormalized = []  # Store all the MC dropout predictions (unnormalized)
    total_mse_unnormalized = 0
    total_mape_unnormalized = 0
    count = 0
    
    for data in dataset:
        x, y = data
        x = x.to(device)
        y = y.to(device)
        mc_preds = []
        
        for _ in range(num_samples):  # Perform `num_samples` stochastic forward passes
            with torch.no_grad():
                pred = model(x)
                mc_pred = pred.cpu().numpy().flatten()
                mc_preds.append(mc_pred)  # Keep predictions normalized initially

        mc_preds = np.array(mc_preds)  # Convert to numpy array for easier calculations

        mc_preds_unnorm = mc_preds * (norm_params['target_max'] - norm_params['target_min']) + norm_params['target_min']
        
        # Mean and standard deviation of the predictions (for confidence intervals)
        mean_pred = mc_preds_unnorm.mean(axis=0)
        std_pred = mc_preds_unnorm.std(axis=0)
        
        true_val_norm = y.cpu().numpy().flatten()

        # Unnormalize for unnormalized metric calculations
        true_val_unnorm = true_val_norm * (norm_params['target_max'] - norm_params['target_min']) + norm_params['target_min']
        mean_pred_unnorm = mean_pred #* (max_stiffness[key] - min_stiffness[key]) + min_stiffness[key]
        std_pred_unnorm = std_pred #* (max_stiffness[key] - min_stiffness[key]) + min_stiffness[key]

        # Calculate MSE using unnormalized values
        mse_unnorm = np.mean((true_val_unnorm - mean_pred_unnorm) ** 2)
        total_mse_unnormalized += mse_unnorm

        # Calculate MAPE using unnormalized values, skipping zero true values
        for t, p in zip(true_val_unnorm, mean_pred_unnorm):
            if t != 0:
                total_mape_unnormalized += np.abs((t - p) / t) * 100
                count += 1

        # Store unnormalized values for plotting or further analysis
        true_values_unnormalized.append(true_val_unnorm)
        mc_predicted_values_unnormalized.append((mean_pred_unnorm, std_pred_unnorm))

    # Final metrics using unnormalized values
    mse = total_mse_unnormalized / len(dataset)
    rmse = np.sqrt(mse)
    avg_mape = total_mape_unnormalized / count if count > 0 else float('inf')
    
    return true_values_unnormalized, mc_predicted_values_unnormalized, mse, rmse, avg_mape



def plot_mc_predictions(true_values, mc_predicted_values, mse, rmse, mape, title, target_indexes=None, confidence_level=1.96, key='df3'):
    """
    Plots the true values, predicted mean values, and confidence intervals.

    Args:
    - true_values: List or array of true values (unnormalized).
    - mc_predicted_values: List of tuples (mean, std) from MC dropout predictions (unnormalized).
    - mse: Mean Squared Error of the predictions (calculated from normalized data).
    - rmse: Root Mean Squared Error of the predictions (calculated from normalized data).
    - mape: Mean Absolute Percentage Error of the predictions (calculated from normalized data).
    - title: Title for the plot.
    - target_indexes: Dictionary with target indexes for adding arrows (optional).
    - confidence_level: z-score for the desired confidence interval (default 1.96 for 95% CI).
    """
    # Extract the mean and std from the predicted values
    mean_pred = np.array([mean for mean, std in mc_predicted_values])
    std_pred = np.array([std for mean, std in mc_predicted_values])

    # Flatten the arrays to ensure they are 1D
    mean_pred = mean_pred.flatten()
    std_pred = std_pred.flatten()

    # Calculate upper and lower confidence bounds
    upper_bound = mean_pred + confidence_level * std_pred
    lower_bound = mean_pred - confidence_level * std_pred

    # Ensure true values are flattened as well
    true_values = np.array(true_values).flatten()


    

    fig_width = 16
    fig_height = 9
    plt.figure(figsize=(fig_width, fig_height))
    
    # Plot true values and mean predictions
    plt.plot(strain_x_rescaled[key], true_values, label="True Values", color="b")
    plt.plot(strain_x_rescaled[key], mean_pred, label="Predicted Mean", color="r", linestyle='--')
    
    # Plot confidence intervals as a shaded region
    plt.fill_between(strain_x_rescaled[key], lower_bound, upper_bound, color="r", alpha=0.3, label=f"Confidence Interval (95%)")


    # Create metrics text with proper alignment using monospace font
    metrics_text = (
        f"MSE:  {mse:.2f} \n"
        f"RMSE: {rmse:.2f} \n"
        f"MAPE: {mape:.2f}%"
    )
    
    # Position the text box in the upper right corner with better styling
    plt.annotate(metrics_text, xy=(0.97, 0.95), xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.6", facecolor='white', alpha=0.8, 
                           edgecolor='gray', linewidth=1),
                 ha='right', va='top', fontsize=18, family='monospace')


    # Increase font sizes
    plt.xlabel("Cycles")
    plt.ylabel("Stiffness (%)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# Run inference on validation and test datasets
true_val, pred_val, val_mse, val_rmse, val_mape = inference(model, val_dataset, device, key=test_key, norm_params=norm_params)

# Plot for validation data
plot_predictions(true_val, pred_val, val_mse, val_rmse, val_mape, title=f"FOD{int(test_key.split('f')[-1])+3} - Cross Validation Fold", key=test_key)

# Plot for test data
#plot_predictions(true_test, pred_test, test_mse, test_rmse, test_mape, title="Test Data: True vs Predicted RUL", key=test_key)

In [ ]:
# Run MC dropout inference on validation and test datasets
true_val, mc_pred_val, val_mse, val_rmse, val_mape = mc_dropout_inference(model, val_dataset, device,norm_params=norm_params, num_samples=100, key=test_key)
#true_test, mc_pred_test, test_mse, test_rmse, test_mape = mc_dropout_inference(model, test_data, device, num_samples=100, key='df3')

# Plot MC predictions for validation data
plot_mc_predictions(true_val, mc_pred_val, val_mse, val_rmse, val_mape, title=f"FOD{int(test_key.split('f')[-1])+3} - Cross Validation Fold", target_indexes=target_indexes[test_key], key=test_key)